In [ ]:
import os
import gc
from glob import glob

import numpy as np
import pandas as pd

from datasets import Dataset, DatasetDict, load_metric

from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, 
                          DataCollatorForSeq2Seq)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
config = {
    "output_dir": "t5_small_lab3_finetune",
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    'num_train_epochs': 4,
    "train_batch_size": 1,
    "eval_batch_size": 1,
    "max_seq_length": 1024,
    "overwrite_output_dir": True,
    "reprocess_input_data": True,
    "fp16": True
}

In [ ]:
prefix_val = "summarize"

df = pd.read_parquet("arxiv_title_generation.parquet")

print("\nDataframe memory usage")
print(df.memory_usage(deep = True))

print(f"Dataframe shape: {df.shape}\n")

print(df.head())

In [ ]:
test_size = 0.2
test_df = df.sample(frac = test_size, random_state = 1970)
train_df = df.drop(index = test_df.index)

print(f"Training instance count: {len(train_df)}\nTest instance count: {len(test_df)}\n")

train_dataset = Dataset.from_dict(train_df)
test_dataset = Dataset.from_dict(test_df)
arxiv_title_dict = DatasetDict({"train": train_dataset,"test": test_dataset})

print(arxiv_title_dict)

In [ ]:
tokenizer_name = "google/t5-efficient-mini"
#tokenizer_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast = False)

def preprocess_function(examples):
    inputs = [f"{prefix_val}: " + doc for doc in examples["input_text"]]
    model_inputs = tokenizer(inputs,
                             max_length = config['max_seq_length'],
                             padding = True,
                             truncation = True)

    labels = tokenizer(text_target = examples["target_text"],
                       max_length = config["max_seq_length"] // 8,
                       padding = True,
                       truncation = True)
    
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_arxiv = arxiv_title_dict.map(preprocess_function, batched = True)

In [ ]:
rouge = load_metric("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(predictions, 
                                           skip_special_tokens = True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(labels, 
                                            skip_special_tokens = True)

    # Compute ROUGE-1 scores
    rouge_scores = rouge.compute(predictions = decoded_preds, 
                                 references = decoded_labels, 
                                 rouge_types = ["rouge1"])["rouge1"]

    # Calculate the mean ROUGE-1 F1 score
    rouge1_f1 = np.mean([score["f"] for score in rouge_scores])

    # Rounds the result to 4 decimal places for cleaner output, and returns it.
    return {"rouge1_f1": round(rouge1_f1, 4)}

In [ ]:
model_type = "google/t5-efficient-mini"
model = AutoModelForSeq2SeqLM.from_pretrained("./t5_small_lab3_finetune/checkpoint-11500")
data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer, model = model_type)

In [ ]:
idx = 100
text = "summarize: " + tokenized_arxiv['test'][idx]['input_text']
actual_text = tokenized_arxiv['test'][idx]['target_text']
print(text)

In [ ]:
from transformers import pipeline

summarizer = pipeline("summarization", model = "./t5_small_lab3_finetune/checkpoint-11500")
pred = summarizer(text)

In [ ]:
print(f"Actual Title: {actual_text}")
print(f"Predicted Title: {pred[0]['summary_text']}")